<a href="https://colab.research.google.com/github/karthiknaiduch/DevTest/blob/main/AI_Agent_RAG_MCP_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Your First AI Agent in 90 Minutes
### The 2026 stack — RAG · MCP · LangGraph

*No fluff. No 2024 tutorials. Just the tools that matter today.*

In one focused session you'll build a **working AI agent** that:

1. **Retrieves from your own data** with a RAG pipeline
2. **Calls tools through the Model Context Protocol (MCP)** — the universal tool standard now adopted by Anthropic, OpenAI, and Google — using an MCP server *you write yourself* in under 20 lines
3. **Orchestrates multi-step reasoning** as a stateful agent with LangGraph

We build **one coherent project**: a support agent for a fictional company, *Northwind Analytics*. By the end it can look up policy in your documents, take real actions through MCP tools, reason across multiple steps, and remember the conversation.

**Stack:** Gemini (`gemini-3.7-flash`) · FAISS · FastMCP 3 · LangGraph 1 · `langchain-mcp-adapters`

**Prereqs:** a laptop, this Colab, and the Gemini API key handed out at the workshop. That's it.

## How to run this notebook

Run the cells **top to bottom** (`Shift + Enter`). Each section builds on the previous one.

| Time | Section | You build |
|------|---------|-----------|
| ~10 min | 0. Setup | API key + a sanity call |
| ~25 min | 1. RAG | A retriever grounded in your documents |
| ~25 min | 2. MCP | Your own MCP server, tested over the protocol |
| ~25 min | 3. LangGraph | The agent that ties RAG + MCP together |
| ~12 min | 3.5 Harness | Budgets, audit logs, and human approval |
| ~8 min | 4. Ship it | Repo files for your GitHub |

> If anything async misbehaves later, use **Runtime → Restart session and run all**. That fixes 90% of live-workshop hiccups.

## 0. Setup

Install the stack. This takes ~1–2 minutes on a fresh Colab.

In [ ]:
# Pinned to the versions this notebook was tested against (Aug 2026).
# langchain 1.x no longer installs langchain-community for you - FAISS needs it explicitly.
%pip install -q -U \
    "langchain>=1.3,<2" "langchain-core>=1.6,<2" "langchain-community>=0.4,<0.5" \
    "langchain-text-splitters>=1.1,<2" "langchain-google-genai>=4.3,<5" \
    "faiss-cpu>=1.15" "langgraph>=1.2,<2" "langchain-mcp-adapters>=0.3,<0.4" \
    "fastmcp>=3.4,<4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.7/224.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.2/237.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4

### Your API key

The workshop key was provided to you. Paste it when prompted (it stays hidden).
If you've saved it in Colab **Secrets** as `GOOGLE_API_KEY`, this cell picks it up automatically.

In [ ]:
import os, sys, getpass

def load_api_key():
    key = None
    try:
        from google.colab import userdata          # Colab Secrets, if set
        key = userdata.get("Workshop")
    except Exception:
        key = None
    if not key:
        key = getpass.getpass("Workshop").strip()
    os.environ["GOOGLE_API_KEY"] = key
    os.environ["GEMINI_API_KEY"] = key            # both names are accepted
    return key

load_api_key()
print("API key set.")

API key set.


### One place to pick our models

We keep model names in variables so you can swap them in a single spot.
`gemini-2.5-flash` is the fast, cheap workhorse and is great for agents.

In [ ]:
CHAT_MODEL  = "gemini-3.7-flash"       # current GA Flash model (Aug 2026)
# CHAT_MODEL = "gemini-2.5-flash"      # fallback: older but still GA, if 3.7 hiccups live

EMBED_MODEL = "gemini-embedding-001"   # stable, 3072-dim
# EMBED_MODEL = "gemini-embedding-2"   # newer multimodal embeddings; NOT wire-compatible
                                       # with 001 - re-embed everything if you switch.

print("Chat model :", CHAT_MODEL)
print("Embed model:", EMBED_MODEL)

Chat model : gemini-3.7-flash
Embed model: gemini-embedding-001


### Sanity check

If this prints a sentence, your key and model work.

> **No `temperature=0` on Gemini 3.x.** Google explicitly recommends leaving temperature at its
> default for Gemini 3 models — setting it below 1.0 can cause looping or degraded reasoning on
> multi-step agent tasks. `langchain-google-genai` already omits it for 3.x unless *you* pass it.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=CHAT_MODEL)   # no temperature - see the note above
print(llm.invoke("In one sentence, what is an AI agent?").content)

[{'type': 'text', 'text': 'An AI agent is an autonomous software system that perceives its environment, makes decisions, and takes actions to achieve a specific goal.', 'extras': {'signature': 'EscLCsQLARFNMg8Ef/leou2OVwdAhALJLd7OZyuKsMqIvWeqH82G7uvReGC5H1UbFBU1pUph6dD/RpEfifJXjgaKxL6Agf/gAEFkle+kLj6eo3SzPNFXwPThfwrlDXmpNtn9FYGHFXuf61smk0vflQ+DxyYiCq6PfJ5xDsrRVdKmLtEn7edUrV2x0JO3NuKgOy3ra6MpEgYkdFxnubOug83cE+TV7115DgQa5CqEBZ+LQ2MUCsvV8984J1Jblok6GPuVciFS13+o6NQndiW+ZklOIWdZO5dCtf8nemTxI0taVrttpidUaDcro3yXWlfQZAS6RD00m3o1jLu0hfI+pXBHS2m2mJLdLZ7jXBWaqbTJhC20AzoN8zR6Uh2K8TF6d3qYdJcuVHH/uChMhK8Wesjtmx+svITKvTGutOuWePW/Hq1q7GhgtW4J87O3t/62RVaEFh5FzGEvybZVplUKy9Gkm0c8dKlzjm9kxVfrSfSxfpCbvmF2kq9ML9awXWIUWs3wenc83V8P/CbDtA/KrlvCp44VNZrtLHiwA2D5K4ljTzZSg6rWOLss+0J6saSJSNthysveTdpAo/V1Uq7mFh1nx9RtTaE6zfPMqmUwxBQgat9cwAdDkAaLZBwfwedl4gsSNbIMkfEt5An+ZDdDHR9kVQCnIKfLt4q0WgciHtBMk3oJ2on1PJ43l6QCzX1sqzvr7Prbd/wjdpIVcAtEgx3xcHe0OaVGx4bE/9tOjgcQSyDb0H8D1FDfMfaeImVhqWErCpQdu6dbM3ZFDBo3XBHt+IIDGMxbulZ0B1

In [ ]:
a = llm.invoke("In one sentence, what is an AI agent?")
a.content[0]['text']

'An AI agent is an autonomous software system powered by artificial intelligence that perceives its environment, makes decisions, and takes actions to achieve specific goals.'

---
## 1. RAG — grounding the agent in your data

An LLM only knows what it was trained on. **Retrieval-Augmented Generation (RAG)** fixes that: we
embed your documents into a vector store, retrieve the most relevant chunks for a question, and hand
*those* to the model as context. The model answers from your data instead of guessing.

The flow: **your docs → chunks → embeddings → vector store → retrieve top-k → answer.**

### Your knowledge base

Below is a small internal knowledge base for *Northwind Analytics*. **This is the "your own data" part** —
edit these strings, or use the optional upload cell to drop in your own `.txt`/`.md` files.

In [ ]:
DOCUMENTS = [
    "Northwind Analytics return policy: customers may return any product within 30 days of "
    "delivery for a full refund, provided they have the original order confirmation email.",

    "Shipping: standard shipping is $5.99 and arrives in 3-5 business days. Orders over $50 "
    "ship free. Expedited shipping is $14.99 and arrives in 1-2 business days.",

    "Accepted payment methods: Visa, Mastercard, American Express, PayPal, Apple Pay, and "
    "Google Pay. We do not accept checks, wire transfers, or cryptocurrency.",

    "Support hours: our team is available Monday through Friday, 9 AM to 6 PM Pacific Time. "
    "Tickets opened outside these hours are answered the next business day.",

    "Warranty: all hardware sensors include a 2-year limited warranty covering manufacturing "
    "defects. The warranty does not cover water damage or accidental drops.",

    "Data retention: customer analytics data is retained for 18 months on the Standard plan and "
    "for 36 months on the Enterprise plan. Customers can request earlier deletion at any time.",

    "Refunds for the annual SaaS subscription are pro-rated and issued to the original payment "
    "method within 7 business days of the cancellation request.",
]

print(f"Loaded {len(DOCUMENTS)} documents.")

Loaded 7 documents.


**OPTIONAL — use your own files.** Uncomment and run to replace the sample docs with your own
text/markdown files, then re-run the cells below this one.

In [ ]:
# from google.colab import files
# uploaded = files.upload()                       # pick .txt / .md files
# DOCUMENTS = [data.decode("utf-8", errors="ignore") for data in uploaded.values()]
# print(f"Loaded {len(DOCUMENTS)} document(s) from upload.")

### Build the vector store

We split the docs into chunks, embed them with Gemini, and index them with FAISS (an in-memory
vector store — no server to run).

> 📎 The import below prints a *"langchain-community is being sunset"* warning. It's expected —
> FAISS still lives there and still works; the package is just in maintenance mode. Ignore it.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.create_documents(DOCUMENTS)
print(f"Split into {len(chunks)} chunks.")

embeddings = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) #topk relevance
print("Vector store ready.")

/tmp/ipykernel_8375/3814978214.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Split into 7 chunks.
Vector store ready.


### Retrieval, on its own

Before we involve the LLM, look at what retrieval returns. Notice it pulls the *relevant* chunks
even though our question doesn't use the exact same words.

In [ ]:
for i, d in enumerate(retriever.invoke("How long do I have to send something back?"), 1):
    print(f"[{i}] {d.page_content}\n")

[1] Northwind Analytics return policy: customers may return any product within 30 days of delivery for a full refund, provided they have the original order confirmation email.

[2] Shipping: standard shipping is $5.99 and arrives in 3-5 business days. Orders over $50 ship free. Expedited shipping is $14.99 and arrives in 1-2 business days.

[3] Support hours: our team is available Monday through Friday, 9 AM to 6 PM Pacific Time. Tickets opened outside these hours are answered the next business day.



### A grounded answer

Now we wire retrieval into the model with LCEL (`|` pipes). The prompt forces the model to answer
**only** from the retrieved context — so it won't invent policy.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the answer isn't in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What payment methods do you accept?"))

Based on the context, the accepted payment methods are:
* Visa
* Mastercard
* American Express
* PayPal
* Apple Pay
* Google Pay


In [ ]:
format_docs(retriever.invoke("What payment methods do you accept?"))

'Accepted payment methods: Visa, Mastercard, American Express, PayPal, Apple Pay, and Google Pay. We do not accept checks, wire transfers, or cryptocurrency.\n\nSupport hours: our team is available Monday through Friday, 9 AM to 6 PM Pacific Time. Tickets opened outside these hours are answered the next business day.\n\nShipping: standard shipping is $5.99 and arrives in 3-5 business days. Orders over $50 ship free. Expedited shipping is $14.99 and arrives in 1-2 business days.'

In [ ]:
RunnablePassthrough().invoke("What payment methods do you accept?")

'What payment methods do you accept?'

**Test the guardrail.** Ask something that isn't in the docs — a grounded system should admit
it doesn't know instead of hallucinating.

In [ ]:
print(rag_chain.invoke("Do you offer same-day delivery by drone?"))

I don't know.


---
## 2. MCP — give your agent tools through a real protocol

RAG lets the agent *read*. To *act* — open a ticket, do a calculation, hit an API — it needs **tools**.

The **Model Context Protocol (MCP)** is the open standard for exposing tools to LLMs. Instead of
hard-wiring functions into one framework, you run a small **MCP server** that any MCP-aware client
(Claude, ChatGPT, your LangGraph agent, an IDE) can discover and call. It was introduced by Anthropic
and is now adopted across the major providers — which is why it's worth learning once and reusing
everywhere.

Three pieces: a **server** (exposes tools), a **client** (calls them), and a **transport** (how they
talk — in-memory, stdio, or HTTP).

You're about to write a server.

### Your MCP server (this is the whole thing)

Decorate plain Python functions with `@mcp.tool`. FastMCP reads the type hints and docstrings and
generates the protocol schema for you. **That's a real, standards-compliant MCP server.**

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("northwind-tools")

@mcp.tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())

@mcp.tool
def create_ticket(summary: str, priority: str = "normal") -> dict:
    """Open a support ticket. Returns the new ticket's id, summary, priority, and status."""
    import random
    return {
        "ticket_id": f"NW-{random.randint(1000, 9999)}",
        "summary": summary,
        "priority": priority,
        "status": "open",
    }

@mcp.tool
def today() -> str:
    """Return today's date in YYYY-MM-DD format."""
    from datetime import date
    return date.today().isoformat()

print("✅ MCP server defined with tools:", [t for t in ("word_count", "create_ticket", "today")])

✅ MCP server defined with tools: ['word_count', 'create_ticket', 'today']


### Call it over the protocol

FastMCP's in-memory client connects straight to your server object — no networking needed — so you
can confirm it speaks MCP correctly right here in the notebook.

In [ ]:
from fastmcp import Client

async def test_server():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        print("Tools the server exposes:", [t.name for t in tools])

        result = await client.call_tool(
            "create_ticket",
            {"summary": "Sensor stopped reporting data", "priority": "high"},
        )
        print("create_ticket ->", result.data)

await test_server()

Tools the server exposes: ['word_count', 'create_ticket', 'today']
create_ticket -> {'ticket_id': 'NW-3998', 'summary': 'Sensor stopped reporting data', 'priority': 'high', 'status': 'open'}


### Save the server to a file

For the agent in Section 3 (and for your GitHub repo), we write the **exact same server** to
`mcp_server.py`. The `mcp.run()` line is what turns it into a standalone process other tools can launch.

In [ ]:
%%writefile mcp_server.py
from fastmcp import FastMCP

mcp = FastMCP("northwind-tools")

@mcp.tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())

@mcp.tool
def create_ticket(summary: str, priority: str = "normal") -> dict:
    """Open a support ticket. Returns the new ticket's id, summary, priority, and status."""
    import random
    return {
        "ticket_id": f"NW-{random.randint(1000, 9999)}",
        "summary": summary,
        "priority": priority,
        "status": "open",
    }

@mcp.tool
def today() -> str:
    """Return today's date in YYYY-MM-DD format."""
    from datetime import date
    return date.today().isoformat()

if __name__ == "__main__":
    mcp.run()   # defaults to stdio transport

Writing mcp_server.py


---
## 3. LangGraph — the agent that ties it all together

So far we have two capabilities sitting side by side: **search** (RAG) and **act** (MCP tools).
An agent decides *which* to use, *in what order*, and *how many times* — looping until the task is done.

**LangGraph** runs that loop as a small state machine and gives us two things we want in production:
**multi-step tool use** and **memory** (a checkpointer that remembers the conversation per thread).

Our agent gets **all three layers at once**: the RAG retriever (as a tool) + your MCP server's tools.

### Wrap RAG as a tool

The agent calls tools, so we expose retrieval as one. (RAG lives in-process here — fast and simple.)

In [ ]:
from langchain_core.tools import tool

@tool
def search_company_docs(query: str) -> str:
    """Search Northwind's internal knowledge base and return the most relevant passages."""
    hits = retriever.invoke(query)
    if not hits:
        return "No relevant documents found."
    return "\n\n".join(f"- {d.page_content}" for d in hits)

print("RAG exposed as a tool:", search_company_docs.name)

RAG exposed as a tool: search_company_docs


### Run your MCP server, then load its tools

Two steps:

1. **Start the server.** We run it with the **Streamable HTTP** transport on a background
   (daemon) thread, so it sits there listening while the rest of the notebook keeps working.
   Streamable HTTP is the current standard transport for networked MCP servers — the older
   "HTTP+SSE" transport is deprecated.
2. **Connect a client.** `langchain-mcp-adapters` speaks MCP to that URL and converts every tool
   it finds into a LangChain tool the agent can call.

> **The whole point of a protocol:** to point this at a *remote* MCP server instead — GitHub,
> Slack, your database — you change the `url` and nothing else. Same tools, same agent code.
> A local server launched as a subprocess is the same story with `"transport": "stdio"` and a
> `command` instead of a `url`.

In [ ]:

import threading, time
from mcp_server import mcp as northwind_server   # the server file we just wrote

MCP_PORT = 8765
MCP_URL  = f"http://127.0.0.1:{MCP_PORT}/mcp"

def _serve():
    northwind_server.run(transport="http", host="127.0.0.1", port=MCP_PORT, show_banner=False)

threading.Thread(target=_serve, daemon=True).start()
time.sleep(4)          # give uvicorn a moment to bind the port
print("MCP server listening at", MCP_URL)

[08/29/26 14:22:16] INFO     Starting MCP server 'northwind-tools' with transport 'http' on        ]8;id=89441;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=194163;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/mixins/transport.py#361\361]8;;\
                             http://127.0.0.1:8765/mcp                                                             

INFO:     Started server process [1131]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)


MCP server listening at http://127.0.0.1:8765/mcp


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "northwind": {
        "transport": "streamable_http",
        "url": MCP_URL,
    }
})

mcp_tools = await mcp_client.get_tools()
print("Loaded MCP tools:", [t.name for t in mcp_tools])

INFO:     127.0.0.1:45794 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:45806 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:45820 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:45822 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:45828 - "DELETE /mcp HTTP/1.1" 200 OK
Loaded MCP tools: ['word_count', 'create_ticket', 'today']


### Assemble the agent

`create_agent` builds the reason-act loop. We give it the model, **all** the tools, a system
prompt that sets its job, and a `MemorySaver` checkpointer for statefulness.

> **API note.** In LangChain 1.x the agent constructor lives at `langchain.agents.create_agent`.
> The old `langgraph.prebuilt.create_react_agent` still runs but is deprecated, and the keyword
> changed: `prompt=` → `system_prompt=`.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

all_tools = [search_company_docs] + mcp_tools

SYSTEM_PROMPT = (
    "You are Northwind Analytics' support agent. "
    "Use search_company_docs to ground every policy answer in the company's documents. "
    "Use the utility tools to take actions, such as opening a ticket. "
    "Never invent policy — if the docs don't cover it, say so."
)

agent = create_agent(
    llm,
    all_tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
)

print("Agent ready with", len(all_tools), "tools:", [t.name for t in all_tools])

Agent ready with 4 tools: ['search_company_docs', 'word_count', 'create_ticket', 'today']


### A helper to talk to it

The agent is async (MCP tools run async), so we `await` it. `thread_id` is the conversation's memory
key — same id means the agent remembers earlier turns.

In [ ]:
async def ask(question, thread="demo-1", show_trace=False):
    config = {"configurable": {"thread_id": thread}}
    out = await agent.ainvoke({"messages": [{"role": "user", "content": question}]}, config)
    if show_trace:
        for m in out["messages"]:
            m.pretty_print()
    else:
        print(out["messages"][-1].content)
    return out

### Watch it use both layers

This question needs RAG (the return policy) **and** an MCP tool (today's date). The agent figures out
it needs both.

In [ ]:
_ = await ask("What's our return policy, and what is today's date?")

INFO:     127.0.0.1:54680 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54682 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:54690 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54692 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54698 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54706 - "DELETE /mcp HTTP/1.1" 200 OK


[{'type': 'text', 'text': "Based on our company documents, here is Northwind Analytics' return and refund policy:\n\n* **Product Returns:** Customers can return any product within **30 days of delivery** for a full refund, provided they have the original order confirmation email.\n* **SaaS Subscriptions:** Refunds for annual SaaS subscriptions are pro-rated and issued to the original payment method within **7 business days** of the cancellation request.\n* **Hardware Warranty:** All hardware sensors include a **2-year limited warranty** covering manufacturing defects (this does not cover water damage or accidental drops).\n\n**Today's date:** August 29, 2026 (2026-08-29).", 'extras': {'signature': 'EokFCoYFARFNMg8MWrYtiVwnAI5xerTGDwdIKkBRBOTlDkhMK732YyhvEmAu4vOjM3whuW8MtuMSgNgUSqewYVnW0gBx/8q+ndP6xYrEi1VW2xczlqyE6S34MOsfKUTjtDmbKm+htCCPBmatrC4/GbOg26qtEH9wglQeufWDR0sfdWCWMi4e9EYo92iIYWOdf6t5CAhPMn3iC9gKVm7n12IXyaBZ/iSsQsV5H3Mc95jKFmcZW6J0t+sK2XflNl4oDOfcaqu1nKDM4mxEb+ivGoXuti5huHj6Jfuf

### Multi-step reasoning + a real action

Now a task that requires *thinking then acting*: check policy, then open a ticket. The agent chains
the tool calls on its own.

In [ ]:
_ = await ask(
    "A customer says their hardware sensor arrived damaged and they're upset. "
    "Check our warranty policy, then open a high-priority ticket summarizing the issue."
)

INFO:     127.0.0.1:32932 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:32946 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:32960 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:32962 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:32968 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:32982 - "DELETE /mcp HTTP/1.1" 200 OK


[{'type': 'text', 'text': '### Warranty Policy Overview\n* **Hardware Sensors:** All hardware sensors come with a **2-year limited warranty** covering manufacturing defects (note: it does not cover water damage or accidental drops).\n* **Return/Replacement Eligibility:** Customers may also return products within **30 days of delivery** for a full refund or replacement, provided they have their original order confirmation email.\n\n---\n\n### Ticket Created\nA high-priority support ticket has been opened for this issue:\n\n* **Ticket ID:** `NW-1781`\n* **Priority:** High\n* **Status:** Open\n* **Summary:** Customer reported hardware sensor arrived damaged upon delivery', 'extras': {'signature': 'EuMCCuACARFNMg/z3pP+xfX8iPUCsNElgRFxE1+k8Atzd0Kf2mmPdIyMJtO20gX7ZWy5tGw9jWwu2s3IWkquFMBDgu5bMj/3Hf826t6WQXaqvekm2h9suJSNqU2gGqeA9fx6vbqQLqSoc0Kz25E24kA38nuHDxGDpUYSGc/Ko1nhoEYu6Ho1TAs4GtrXetUENf4i8b9zej7+nh4Uf7+S4p7XS1L7dq4qsxexBPO6N5xC1C4dPJwZXM197DgpaQzExk8PqO2YsPGyLZEsQGTG2LBOoPLgVgASaJv48hMi

### Statefulness — it remembers

Same `thread_id`, so the agent recalls the ticket it just opened. No re-explaining.

In [ ]:
_ = await ask("What priority did you set on that ticket?")

[{'type': 'text', 'text': 'The ticket was set to **high** priority.', 'extras': {'signature': 'EqcCCqQCARFNMg/wQkQNCiZVGebEVwbaMYja8VEYNanw/h5iHCZZvN3gVS1lLbZbLWX+m6+AE1xABG5OGKdAu3vJrIqFSbix+eECZkM6XobJjL8L0Ug5bmD25G8vN5X/U/Q9WLUfK1OmrGzj0Usn7RcTpaFuX97u23PqEP696OxLeTU2dg2sV+ACIRYk4kxN/OwzVkx5rhfD+digc1IqR67YjWKU+ypIdvycrDaWQMUOiHgUBCJzTvKWpMFKIoYwYkB5a3ILf1w5T5V72Yqa+2dkKy1Sa9J5Us/aJsq8binIUKZSBzLF6M1JC/CE+5a+VkL+Bbld6ujjNCtpak/02QevG0BGkhMcKVyQ4MntO2l9PYrDp7bXdvTu66IR2NHIKT1D7FcBah85RQ=='}}]


A **different** `thread_id` is a clean slate — proof that memory is scoped per conversation, not global.

In [ ]:
_ = await ask("What ticket did we just open?", thread="brand-new-conversation")

[{'type': 'text', 'text': "No support ticket has been opened yet during our current conversation. \n\nIf you would like me to open a ticket for you, please let me know the issue summary/details and your preferred priority (e.g., Low, Medium, High, Urgent), and I'll be happy to create one!", 'extras': {'signature': 'EqYHCqMHARFNMg8cMsEfXaGBKwUresIDY0MwHkCYh90URiqTTFQjsdrFrqvaxNfMp9HEID8y9eAH5ZaS+5uQtTRmbU25zWfAyCCL6VxtQ5dBqSVMxgo+5Eo04PIdhMvUPGuInhZHoJ7hEoUtxprf2YakUVJul+XRJjb4wKu3SLkHCZshweRcMtwcGWZJOMveA0lDdsxoF37+zvw/ZTO8Dglq3qbQoE3vhIMg8MsW1/J9Ml6zOBu+0zbV078Um8ZUGApsi1Lz69bqxemxu7byFUt8pJQLeeUaqSFhku3nFic/qQOu7a1WgYtwr/lUJfB8bq5RE1no6Ky5Uv1DudoTcg0rj8h5nO/MCoK0ZTzs8WKZKOrlzKWRVRBzsMp1StWts5Y+27GbBrTHdyw6g+0PtuDrsIDh9+py8dip/Jc0Mgs0w/74cwMCRQAt6XM4qn5Unq95Oc242s8TK5WNt1KMwhCFGPwXE2ZhRoCbYo8l2cNWlVMFo+Rzbriy+Mwmj6V1XaS8767t0+GXUiODxwU72L39Gr+XsTdrDtb5avH8Pc+eD4jVFQlEcpZ+JTr9MeDTthaZYEuljBJrjf5n2r001/BtulxAmc0kHH/Xnbq0sRx2cNHOcKECahE77Njta8nbewOvM0ofSuWGOPo5JgQbYLb7iWFp9YGn0Ld9WuEdhCX

### See it think

Set `show_trace=True` to print the full reason-act trace: the agent's tool calls, the tool outputs,
and the final answer. This is how you debug agents.

In [ ]:
_ = await ask("How much is shipping on a $40 order, and how much would I save by hitting $50?",
              thread="trace", show_trace=True)

================================ Human Message =================================

How much is shipping on a $40 order, and how much would I save by hitting $50?
================================== Ai Message ==================================

[]
Tool Calls:
  search_company_docs (call_289671)
 Call ID: call_289671
  Args:
    query: shipping cost threshold policy
================================= Tool Message =================================
Name: search_company_docs

- Shipping: standard shipping is $5.99 and arrives in 3-5 business days. Orders over $50 ship free. Expedited shipping is $14.99 and arrives in 1-2 business days.

- Northwind Analytics return policy: customers may return any product within 30 days of delivery for a full refund, provided they have the original order confirmation email.

- Support hours: our team is available Monday through Friday, 9 AM to 6 PM Pacific Time. Tickets opened outside these hours are answered the next business day.
===========================

---
## 3.5 The harness — turning the loop into something you would deploy

You have a working agent. It is not yet a working *product*.

`create_agent` gives you a loop. A loop will happily call a tool forty times, spend your budget
at 3am, and open a customer ticket nobody approved. The distance between the two is not a better
model or a better prompt — it is a **harness**: budgets, approval, retries, redaction, logging.

In LangChain 1.x that harness is **middleware**, and it composes. You keep the agent you already
built and wrap it.

### Where middleware plugs in

Middleware wraps the agent loop at six points. You do not need all six today — you need the shape,
so you can look up the rest.

| Hook | When it runs | Typical use |
|---|---|---|
| `before_agent` | once per invocation | load memory, validate input |
| `before_model` | before each model call | trim history, catch sensitive data |
| `wrap_model_call` | around the model call | caching, retries, swapping the model |
| `wrap_tool_call` | around each tool execution | audit, gate, intercept results |
| `after_model` | after each model response | approval gates, output checks |
| `after_agent` | once at the end | cleanup, notification |

Everything below is one import and one entry in a `middleware=[...]` list.

### 3.5a — An audit log, in six lines

The clearest possible proof of this morning's boundary: **the model never touches your data.**
It proposes; your code executes. `wrap_tool_call` sits exactly in that gap, so it sees every
proposal before it runs and every result before the model does.

In [ ]:
from langchain.agents.middleware import wrap_tool_call, ToolCallRequest

@wrap_tool_call
async def audit(request: ToolCallRequest, handler):
    print(f"[audit] CALL {request.tool_call['name']} args={request.tool_call['args']}")
    result = await handler(request)          # this line is where the tool actually runs
    print(f"[audit] ->   {result.content[:70]}")
    return result

print("✅ audit middleware defined")

**Note the `async`.** Our agent is async because MCP tools are async. A `@wrap_tool_call` written
as a plain `def` raises `NotImplementedError` the moment you `ainvoke`. Wrap-style hooks must match
the path they run on; node-style hooks like `@before_model` work either way.

In [ ]:
audited = create_agent(
    llm, all_tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
    middleware=[audit],                      # <- the only new argument
)

_ = await audited.ainvoke(
    {"messages": [{"role": "user",
                   "content": "Check our warranty policy, then open a high-priority ticket "
                              "for a sensor that arrived damaged."}]},
    {"configurable": {"thread_id": "audit-demo"}},
)

Every tool call and every result printed, in order, from outside the model. That log is what an
auditor asks for — and you did not modify the agent, the tools, or the prompt to get it.

### 3.5b — A budget ceiling

An agent decides how many steps to take. Left alone, "how many" can be "a lot". A ceiling is two
lines, and it belongs in every agent you deploy.

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

capped = create_agent(
    llm, all_tools,
    system_prompt=SYSTEM_PROMPT,
    middleware=[ToolCallLimitMiddleware(run_limit=2)],   # at most 2 tool calls per question
)

out = await capped.ainvoke({"messages": [{"role": "user", "content":
    "Look up our return policy, our warranty, our shipping cost and our payment methods, "
    "then open a ticket summarising all four."}]})

for m in out["messages"]:
    print(f"{type(m).__name__:<13} | {(m.content or '')[:72]}")

Look at what happens at the ceiling. The run does **not** crash and does **not** silently stop —
the blocked calls come back as a `ToolMessage` reading *"Tool call limit exceeded"*, which goes
into the model's context, so the model reads it and wraps up gracefully.

That is the pattern worth stealing: **a good guardrail tells the model what happened rather than
pulling the plug.**

### 3.5c — Human approval before it does something real

Some tools read. Some tools change the world. `search_company_docs` is safe to run a hundred times;
`create_ticket` writes to a system of record and a human may want a say.

`HumanInTheLoopMiddleware` pauses the run *before* a named tool executes and hands control back to
you. Because we have a checkpointer, that pause can last a millisecond or three days.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

gated = create_agent(
    llm, all_tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),                        # required: the pause needs somewhere to live
    middleware=[HumanInTheLoopMiddleware(interrupt_on={
        "create_ticket":       {"allowed_decisions": ["approve", "edit", "reject"]},
        "search_company_docs": False,                  # read-only: never interrupt
    })],
)

cfg = {"configurable": {"thread_id": "approval-demo"}}
res = await gated.ainvoke({"messages": [{"role": "user", "content":
    "The customer's sensor arrived damaged. Check the warranty, then open a high-priority ticket."}]}, cfg)

req = res["__interrupt__"][0].value["action_requests"][0]
print("PAUSED — the agent wants to run:")
print("  tool:", req["name"])
print("  args:", req["args"])

Notice what already happened: the agent searched the docs freely, then stopped dead at the write.
Now you decide. Approve it:

In [ ]:
out = await gated.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg)
print(out["messages"][-1].content)

Or refuse, with a reason the model can read and respond to:

```python
Command(resume={"decisions": [{
    "type": "reject",
    "message": "The customer has not confirmed the damage yet — ask them first."
}]})
```

Or edit the arguments before it runs:

```python
Command(resume={"decisions": [{
    "type": "edit",
    "edited_action": {"name": "create_ticket",
                      "args": {"summary": "Sensor arrived damaged", "priority": "normal"}}
}]})
```

> **The rule to take home:** gate the tools that **write**. Reading a document, looking up a record,
> doing arithmetic — let the agent run. Sending an email, issuing a refund, deleting a row —
> approval, or a hard constraint outside the model. **Autonomy should be proportional to
> reversibility.**

### What else ships in the box

You will not need these today. You will need them at work, and each is one import.

| Middleware | What it does |
|---|---|
| `ToolRetryMiddleware` | retry flaky tools with exponential backoff |
| `ModelRetryMiddleware` | the same for rate limits and timeouts |
| `ModelFallbackMiddleware` | fall back to another model when the primary fails |
| `SummarizationMiddleware` | compress old turns as the conversation grows |
| `ContextEditingMiddleware` | drop stale tool outputs, keep the recent ones |
| `PIIMiddleware` | redact, mask, hash or block emails, cards, custom patterns |
| `ModelCallLimitMiddleware` | a ceiling on model calls per run or per thread |
| `LLMToolSelectorMiddleware` | with many tools, pre-select the relevant few |
| `TodoListMiddleware` | give the agent a task list for long multi-step work |
| `LLMToolEmulator` | fake tool results for testing, with no side effects |

`SummarizationMiddleware` and `ContextEditingMiddleware` are both answers to a cost problem you
already saw: every step resends the whole transcript. That is **context engineering**, with an
import statement.

### Your turn (5 minutes)

Stack all three onto one agent, and add one rule of your own.

1. Build an agent with `audit`, a `ToolCallLimitMiddleware(run_limit=4)`, **and** approval on
   `create_ticket`.
2. Add a `@before_model` hook that prints how many messages are in context on each model call —
   watch it grow, and connect that to the cost conversation.
3. Ask it something that needs two searches and a ticket. Approve the ticket.

In [ ]:
# YOUR TURN — build it here.
# Hints:
#   from langchain.agents.middleware import before_model, AgentState
#   from langgraph.runtime import Runtime
#   @before_model
#   def show_context(state: AgentState, runtime: Runtime):
#       ...
#   middleware=[...] takes a list — order matters, outermost first.

---
## 4. Ship it — leave with a repo

You built a working agent. Let's make it a repo you can push to GitHub.

### `requirements.txt`

In [ ]:
%%writefile requirements.txt
langchain>=1.3,<2
langchain-core>=1.6,<2
langchain-community>=0.4,<0.5
langchain-text-splitters>=1.1,<2
langchain-google-genai>=4.3,<5
faiss-cpu>=1.15
langgraph>=1.2,<2
langchain-mcp-adapters>=0.3,<0.4
fastmcp>=3.4,<4

### `README.md`

In [ ]:
%%writefile README.md
# Northwind Support Agent

A small AI agent built on the 2026 stack: **RAG + MCP + LangGraph**.

- **RAG** — answers grounded in company documents (FAISS + Gemini embeddings)
- **MCP** — tools served by a FastMCP server (`mcp_server.py`), over Streamable HTTP
- **LangGraph** — a stateful reason-act agent that combines both

## Run

```bash
pip install -r requirements.txt
export GOOGLE_API_KEY=your_key_here
```

Open the notebook, or run the MCP server on its own:

```bash
python mcp_server.py                      # stdio, for Claude Desktop / CLI clients
fastmcp run mcp_server.py --transport http --port 8765   # as a web service
```

## Stack
Gemini `gemini-3.7-flash` · `gemini-embedding-001` · FAISS · FastMCP 3 · LangGraph 1 · langchain-mcp-adapters

### Push to GitHub

From a terminal where `mcp_server.py`, `requirements.txt`, and `README.md` live
(download them from Colab's file browser, or `git init` in your own clone):

```bash
git init
git add mcp_server.py requirements.txt README.md
git commit -m "My first RAG + MCP + LangGraph agent"
git branch -M main
git remote add origin https://github.com/<you>/northwind-agent.git
git push -u origin main
```

> In Colab, open the **Files** panel (folder icon, left) to download the three files we wrote.

## What you built

- A **RAG** pipeline grounded in your own documents
- Your own **MCP server** in Python — tested over the real protocol
- A **stateful LangGraph agent** that retrieves, acts, reasons across steps, and remembers
- A **harness** around it: an audit log, a tool-call budget, and human approval on writes

### Where to go next
- **FastMCP** — resources, prompts, and HTTP transport for real deployments: <https://gofastmcp.com>
- **LangGraph** — custom graphs, human-in-the-loop, streaming: <https://langchain-ai.github.io/langgraph/>
- **Gemini API** — models, function calling, embeddings: <https://ai.google.dev/gemini-api/docs>
- **MCP** — the spec and the growing ecosystem of servers: <https://modelcontextprotocol.io>
- **Agent middleware** — the full catalogue of prebuilt hooks: <https://docs.langchain.com/oss/python/langchain/middleware/built-in>

You now have the current stack in your head — and an agent on your GitHub. 🚀